# Conceptual Pre-requisites for `Canonicalizing Open Knowledge Bases` Paper

## 1. Knowledge Bases

### What Is A Knowledge Base?
In layman terms, a `Knowledge Base` is a structured way to represent facts about the world.

Imagine on one hand we have a book which has facts and sentences written in the random way and every knowledge is scattered. On the other hand we have a notebook where we write down the facts like:
- "Barack Obama was born in Honolulu"
- "Apple Inc. was founded by Steve Jobs"

Instead of random sentences we write these facts in a machine-readable form:
```text
(Barack Obama, place of birth, Honolulu)
(Apple Inc., founded by, Steve Jobs)
```

Here each fact is called a triple: `(subject, predicate, object)`.


### Core Components of a KB

#### 1. Triples (aka Facts)

Form:

```
(subject, predicate, object)  
= (s, p, o)
```

Example:

```
("Paris", "isCapitalOf", "France")
```

Each triple asserts a relationship.

#### 2. Entities
* **Subject/Object**: real-world items (people, places, things)
* Stored with unique IDs, e.g., Freebase ID `/m/06cx9` for Barack Obama

#### 3. Relations / Predicates
* The link between subject and object, e.g., `bornIn`, `hasChild`, `presidentOf`



### Types of Knowledge Bases
| Type             | Description                                        | Example           |
| ---------------- | -------------------------------------------------- | ----------------- |
| Open KB          | Facts are extracted from text, no strict schema    | ReVerb, OpenIE    |
| Closed KB        | Schema-defined with predefined relations and types | Freebase, DBpedia |
| Probabilistic KB | Stores facts with uncertainty/confidence           | Knowledge Vault   |


### Formal Foundation

####  KB as a Set

Let:

* 𝔼 = set of entities
* ℛ = set of relations
* 𝔽 = set of facts = subset of 𝔼 × ℛ × 𝔼

Then:

```
KB = { (s, r, o) ∈ 𝔼 × ℛ × 𝔼 }
```

#### Logical View

Each triple (s, r, o) is equivalent to a first-order logic assertion:

```
r(s, o)
```

e.g.,

```
bornIn(BarackObama, Honolulu)
```



### Graph Representation

A KB is naturally a **directed labeled multigraph**:

* Nodes = entities
* Edges = labeled by predicates

Example:

```
Barack Obama ──bornIn──▶ Honolulu
```

> We will use libraries like networkx in Python to model this.

In [1]:
import networkx as nx

G = nx.DiGraph()
G.add_edge("Barack Obama", "Honolulu", relation="bornIn")

In [5]:
class KnowledgeBase:
    def __init__(self):
        self.triples = set() # A set to store unique triples

    def add_fact(self, subject, predicate, object_):
        self.triples.add((subject, predicate, object_)) # Storing as a tuple

    def query(self, subject=None, predicate=None, object_=None):
        return {
            (s, p, o)
            for s, p, o in self.triples # Iterate through stored triples
            if (subject in [None, s]) and # Check if subject matches or is None
               (predicate in [None, p]) and # Check if predicate matches or is None
               (object_ in [None, o]) # Check if object matches or is None
        }


# Example Usage
kb = KnowledgeBase()
kb.add_fact("Obama", "bornIn", "Honolulu")
kb.add_fact("Obama", "presidentOf", "USA")

# Query all facts where subject is Obama
print(kb.query(subject="Obama"))

{('Obama', 'bornIn', 'Honolulu'), ('Obama', 'presidentOf', 'USA')}


### RDF – Resource Description Framework

### Intuition: What is RDF?
RDF is a framework for representing structured data as triples:
```text
(subject, predicate, object)
```
We can think of it as:
* **Subject**: a thing we’re describing
* **Predicate**: a property or relationship
* **Object**: the value or entity connected to the subject

#### Real-World Analogy

Imagine that we have a filing cabinet where every card reads like:

```
(Mona Lisa, paintedBy, Leonardo da Vinci)
(Leonardo da Vinci, bornIn, Vinci)
(Mona Lisa, locatedIn, Louvre)
```

This "card catalog" is the core idea of RDF: **facts represented as triples**.

---

#### Core Concepts and Vocabulary

RDF is part of the **Semantic Web** and uses Internationalized Resource Identifiers(IRIs) instead of simple names.

#### RDF Triple Components

| Component | Role                      | Example                                     |
| --------- | ------------------------- | ------------------------------------------- |
| Subject   | Entity/resource           | `<http://example.org/MonaLisa>`             |
| Predicate | Property/relationship     | `<http://example.org/paintedBy>`            |
| Object    | Value (entity or literal) | `<http://example.org/Leonardo>` or `"1503"` |

---

#### Formal Semantics and Model-Theoretic Foundation

#### RDF Graph

Mathematically, an RDF graph is a **set of triples**:
Let:

* **U** be the set of all URIs (IRIs)
* **L** be the set of all literals (strings, numbers, etc.)
* **B** be a set of blank (anonymous) nodes

Each triple:

```
(s, p, o) ∈ (U ∪ B) × U × (U ∪ B ∪ L)
```

This allows:

* **s** = URI or blank node
* **p** = URI only (predicates must be defined)
* **o** = URI, blank node, or literal

#### Graphical View

Triples form a **directed labeled graph**, with edges = predicates.

```
[Mona Lisa] ──paintedBy──▶ [Leonardo]
[Mona Lisa] ──year──────▶ "1503"
```

### 3Entailment and Semantics

Given:

```
(:x, :p, :y), (:y, :p, :z)
```

We can define inference rules. For instance, if `p` is transitive:

```
⇒ (:x, :p, :z)
```

This forms the basis of **RDFS** and **OWL** (RDF Schema and Web Ontology Language), extending RDF to support richer logic.

#### Creating RDF In Python

In [6]:

from rdflib import Graph, URIRef, Literal, Namespace

ex = Namespace("http://example.org/") # Defining a namespace for oour RDF dat
g = Graph()

g.add((ex.MonaLisa, ex.paintedBy, ex.Leonardo)) # Adding a triple to the graph
g.add((ex.MonaLisa, ex.year, Literal("1503"))) # Adding another triple with a literal value

# Print all triples
for s, p, o in g:
    print(s, p, o)

http://example.org/MonaLisa http://example.org/paintedBy http://example.org/Leonardo
http://example.org/MonaLisa http://example.org/year 1503


### SPARQL – RDF Query Language

#### Intuition: What is SPARQL?

**SPARQL** is to RDF what **SQL** is to relational databases. It lets us **query RDF graphs** by **matching patterns**.

### Real-World Analogy

In a library, we might ask:

> "Find all paintings created by Leonardo."

In SPARQL:

```sparql
SELECT ?painting
WHERE {
  ?painting <http://example.org/paintedBy> <http://example.org/Leonardo> .
}
```

---

#### SPARQL Query Structure

#### Basic Syntax

```sparql
PREFIX ex: <http://example.org/>

SELECT ?painting ?year
WHERE {
  ?painting ex:paintedBy ex:Leonardo .
  ?painting ex:year ?year .
}
```

* `?var` = query variable
* Each clause in `WHERE` = triple pattern
* Matches data like regex matches strings

#### Filters and Conditions

```sparql
FILTER (?year > "1500")
```

#### Optional Matches

```sparql
OPTIONAL { ?painting ex:location ?loc }
```

#### Aggregates and Grouping

```sparql
SELECT ?artist (COUNT(?painting) AS ?count)
WHERE {
  ?painting ex:paintedBy ?artist .
}
GROUP BY ?artist
```

#### Mathematical View: Pattern Matching

Let:

* G = RDF graph = set of triples
* P = Basic Graph Pattern (BGP) = set of triple patterns with variables

The **evaluation** of BGP `P` over `G` is:

```
eval(P, G) = { μ | dom(μ) = vars(P), and μ(P) ⊆ G }
```

Where:

* `μ` = variable mapping (binding)
* `μ(P)` = set of triples produced by substituting variables

SPARQL answers are derived by evaluating these mappings over the graph.